In [50]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [51]:
# reading words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))

32033
15


In [52]:
# Encoding and decoding
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0

itos = {s:i for i, s in stoi.items()}
vocab_size = len(itos)
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [53]:
# Shuffling words
import random
random.seed(42)
random.shuffle(words)

In [54]:
# Build dataset
block_size = 3

def build_dataset(words):

    X, Y = [], []

    for word in words:
        context = [0] * block_size
        for ch in word + '.': 
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)

    print(X.shape, Y.shape)
    return X, Y


n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [55]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

... --> y
..y --> u
.yu --> h
yuh --> e
uhe --> n
hen --> g
eng --> .
... --> d
..d --> i
.di --> o
dio --> n
ion --> d
ond --> r
ndr --> e
dre --> .
... --> x
..x --> a
.xa --> v
xav --> i
avi --> e


In [56]:
# Liner, BatchNorm for experiment optimization

class Linear:

    def __init__(self, fan_in, fan_out, bias = True):
        self.weight = (torch.randn(fan_in, fan_out)) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([self.bias if self.bias is not None else []])
    

class BatchNorm:

    def __init__(self, dim, eps = 1e-5, momentum = 0.1):

        self.dim = dim
        self.eps = eps
        self.momentum = momentum
        self.traning = True

        # Parameters (changes with traning)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Buffers (changes with momentum)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        
        # Calculates forward pass
        if self.traning:
            x_mean = x.mean(0,keepdim=True)
            x_var = x.var(0, keepdim=True)
        else:
            x_mean = self.running_mean
            x_var = self.running_var

        xhat = (x-x_mean) / torch.sqrt(x_var + self.eps)
        self.out = self.gamma * xhat + self.beta

        # Update the buffers
        if self.traning:
            with torch.no_grad():
                self.running_mean = ((1 - self.momentum) * self.running_mean) + (self.momentum * x_mean)
                self.running_var = ((1-self.momentum) * self.running_var) + (self.momentum * x_var)
            return self.out
        
    def parameters(self):
            return [self.gamma, self.beta]
        

class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return[]
                

In [57]:
# Manually setting seed

torch.manual_seed(42)

In [ ]:
# Building the nn layers

n_embed = 10
n_hidden = 200

C = torch.randn((vocab_size, n_embed))

layers = [
    Linear(n_embed * block_size, n_hidden, bias = False),
    BatchNorm(n_hidden),
    Tanh(),
    Linear(n_hidden, vocab_size),
]

# Parameter initialization
with torch.no_grad():
    layers[-1].weight *= 0.1

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

AttributeError: 'list' object has no attribute 'nelement'